In [1]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 1}
import datetime as dt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

import warnings
warnings.simplefilter(action="ignore")

pd.set_option('display.max_columns',1000)
pd.set_option('display.width', 500)
pd.set_option('display.float_format',lambda x : '%.2f' % x)

In [2]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 2}
df_ = pd.read_csv("data/dataset.csv", compression="gzip")
df = df_.copy()
df.head()

,RecipeId,Name,CookTime,PrepTime,TotalTime,RecipeIngredientParts,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeInstructions
0,38,Low-Fat Berry Blue Frozen Dessert,1440,45,1485,"c(""blueberries"", ""granulated sugar"", ""vanilla ...",170.90,2.50,1.30,8.00,29.80,37.10,3.60,30.20,3.20,"c(""Toss 2 cups berries with sugar."", ""Let stan..."
1,41,Carina's Tofu-Vegetable Kebabs,20,1440,1460,"c(""extra firm tofu"", ""eggplant"", ""zucchini"", ""...",536.10,24.00,3.80,0.00,1558.60,64.20,17.30,32.10,29.30,"c(""Drain the tofu, carefully squeezing out exc..."
2,42,Cabbage Soup,30,20,50,"c(""plain tomato juice"", ""cabbage"", ""onion"", ""c...",103.60,0.40,0.10,0.00,959.30,25.10,4.80,17.70,4.30,"c(""Mix everything together and bring to a boil..."
3,45,Buttermilk Pie With Gingersnap Crumb Crust,50,30,80,"c(""sugar"", ""margarine"", ""egg"", ""flour"", ""salt""...",228.00,7.10,1.70,24.50,281.80,37.50,0.50,24.70,4.20,"c(""Preheat oven to 350°F."", ""Make pie crust, u..."
4,46,A Jad - Cucumber Pickle,0,25,25,"c(""rice vinegar"", ""haeo"")",4.30,0.00,0.00,0.00,0.70,1.10,0.20,0.20,0.10,"c(""Slice the cucumber in four lengthwise, then..."


In [3]:
# --- [CELL 2]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 3}
def grab_col_names(dataframe, cat_th=10, car_th=20):

    cat_cols = [col for col in dataframe.columns if dataframe[col].dtypes == "O"]
    num_but_cat = [col for col in dataframe.columns if dataframe[col].nunique() < cat_th and
                   dataframe[col].dtypes != "O"]
    cat_but_car = [col for col in dataframe.columns if dataframe[col].nunique() > car_th and
                   dataframe[col].dtypes == "O"]
    cat_cols = cat_cols + num_but_cat
    cat_cols = [col for col in cat_cols if col not in cat_but_car]

    # num_cols
    num_cols = [col for col in dataframe.columns if dataframe[col].dtypes != "O"]
    num_cols = [col for col in num_cols if col not in num_but_cat]

    print(f"Observations: {dataframe.shape[0]}")
    print(f"Variables: {dataframe.shape[1]}")
    print(f'cat_cols: {len(cat_cols)}')
    print(f'num_cols: {len(num_cols)}')
    print(f'cat_but_car: {len(cat_but_car)}')
    print(f'num_but_cat: {len(num_but_cat)}')
    return cat_cols, num_cols, cat_but_car

In [4]:
# --- [CELL 3]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 4}
cat_cols, num_cols, num_but_cat = grab_col_names(df)

Observations: 375703
Variables: 16
cat_cols: 0
num_cols: 13
cat_but_car: 3
num_but_cat: 0


In [5]:
# --- [CELL 4]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 5}
def outlier_thresholds(dataframe, col_name, q1=0.01, q3=0.99):
    quartile1= dataframe[col_name].quantile(q1)
    quartile3= dataframe[col_name].quantile(q3)
    interquantile_range = quartile3 -quartile1
    up_limit= quartile3 +1.5 * interquantile_range
    low_limit= quartile1 -1.5 * interquantile_range
    return low_limit, up_limit

In [6]:
# --- [CELL 5]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 6}
def replace_with_thresholds(dataframe, variable):
    low_limit, up_limit = outlier_thresholds(dataframe, variable)
    dataframe.loc[(dataframe[variable] < low_limit), variable] = low_limit
    dataframe.loc[(dataframe[variable] > up_limit), variable] = up_limit

for col in num_cols:
    replace_with_thresholds(df, col)

In [7]:
# --- [CELL 6]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 7}
def check_outlier(dataframe, col_name):
    low_limit, up_limit = outlier_thresholds(dataframe, col_name)
    if dataframe[(dataframe[col_name] > up_limit) | (dataframe[col_name] < low_limit)].any(axis=None):
        return True
    else:
        return False

check_outlier(df,num_cols)

False

In [8]:
# --- [CELL 7]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 8}
df= df.iloc[:,1:]

In [9]:
# --- [CELL 8]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 9}
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from yellowbrick.cluster import KElbowVisualizer
from scipy.cluster.hierarchy import linkage
from scipy.cluster.hierarchy import dendrogram
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import AgglomerativeClustering

In [10]:
# --- [CELL 9]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 10}
cat_cols, num_cols, num_but_cat = grab_col_names(df)

Observations: 375703
Variables: 15
cat_cols: 0
num_cols: 12
cat_but_car: 3
num_but_cat: 0


In [11]:
# --- [CELL 10]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 11}
df2=df.copy()

In [12]:
# --- [CELL 11]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 12}
sc = MinMaxScaler((0, 1))
df2[num_cols] = sc.fit_transform(df2[num_cols])

In [13]:
# --- [CELL 12]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 13}
kmeans = KMeans(n_clusters=30, n_init="auto").fit(df2[["TotalTime","Calories","SugarContent"]])

In [14]:
# --- [CELL 13]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 14}
clusters_kmeans = kmeans.labels_
clusters_kmeans

array([20, 20, 17, ..., 15,  9, 29], dtype=int32)

In [15]:
# --- [CELL 14]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 15}
df["kmeans_cluster"] = clusters_kmeans
df["kmeans_cluster"]= df["kmeans_cluster"] + 1
df.head()

,Name,CookTime,PrepTime,TotalTime,RecipeIngredientParts,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeInstructions,kmeans_cluster
0,Low-Fat Berry Blue Frozen Dessert,1200,45,1485.00,"c(""blueberries"", ""granulated sugar"", ""vanilla ...",170.90,2.50,1.30,8.00,29.80,37.10,3.60,30.20,3.20,"c(""Toss 2 cups berries with sugar."", ""Let stan...",21
1,Carina's Tofu-Vegetable Kebabs,20,600,1460.00,"c(""extra firm tofu"", ""eggplant"", ""zucchini"", ""...",536.10,24.00,3.80,0.00,1558.60,64.20,17.30,32.10,29.30,"c(""Drain the tofu, carefully squeezing out exc...",21
2,Cabbage Soup,30,20,50.00,"c(""plain tomato juice"", ""cabbage"", ""onion"", ""c...",103.60,0.40,0.10,0.00,959.30,25.10,4.80,17.70,4.30,"c(""Mix everything together and bring to a boil...",18
3,Buttermilk Pie With Gingersnap Crumb Crust,50,30,80.00,"c(""sugar"", ""margarine"", ""egg"", ""flour"", ""salt""...",228.00,7.10,1.70,24.50,281.80,37.50,0.50,24.70,4.20,"c(""Preheat oven to 350°F."", ""Make pie crust, u...",16
4,A Jad - Cucumber Pickle,0,25,25.00,"c(""rice vinegar"", ""haeo"")",4.30,0.00,0.00,0.00,0.70,1.10,0.20,0.20,0.10,"c(""Slice the cucumber in four lengthwise, then...",30


In [16]:
# --- [CELL 15]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 16}
# === BEFORE (original) ===
# df.groupby('kmeans_cluster').agg({1: ['count','mean', 'median', 'sum'],
#                                     2: ['count','mean', 'median', 'sum'],
#                                     3: ['count','mean', 'median', 'sum'],
#                                     4: ['count','mean','median', 'sum']})

# === AFTER (edited) ===
df.groupby('kmeans_cluster').agg({"TotalTime": ['count','mean', 'median', 'sum'],
                                "Calories": ['count','mean', 'median', 'sum'],
                                "SugarContent": ['count','mean', 'median', 'sum']})

TotalTime                            Calories                           SugarContent                       
                   count    mean  median        sum    count   mean median         sum        count  mean median       sum
kmeans_cluster                                                                                                            
1                  16842   42.66   32.00  718495.00    16842 178.07 175.40  2998984.10        16842 13.75  13.70 231556.20
2                  31016   42.88   35.00 1330078.00    31016 294.70 291.30  9140559.40        31016  1.72   1.80  53335.80
3                   9517   55.05   45.00  523903.00     9517 311.90 317.10  2968330.60         9517 31.23  31.20 297234.40
4                    760 1569.45 1470.00 1192780.00      760 275.97 227.20   209737.80          760 14.12  13.60  10735.00
5                  14389   58.76   45.00  845518.00    14389 531.21 518.90  7643555.20        14389  7.34   7.20 105619.40
6                   5817   58.29   45.00  339078.00     5817 481.14 462.70  2798771.70         5817 19.59  19.40 113932.80
7                  17235   41.61   30.00  717214.00    17235 141.62 139.10  2440814.10        17235 10.27  10.20 176985.10
8                   9780   55.72   40.00  544965.00     9780 629.13 615.60  6152875.80         9780  3.03   3.20  29661.60
9                  32824   36.43   30.00 1195669.00    32824 174.89 174.70  5740640.20        32824  1.15   1.10  37888.00
10                 11969   46.68   35.00  558698.00    11969 226.15 230.00  2706748.90        11969 21.97  21.90 262996.30
11                  1930 1601.28 1500.00 3090469.50     1930 225.10 172.85   434449.00         1930  2.66   2.05   5138.90
12                  4655  533.62  495.00 2484021.00     4655 318.12 319.50  1480840.00         4655  3.47   3.40  16140.70
13                  6023   56.75   40.00  341812.00     6023 285.39 296.40  1718903.10         6023 36.73  36.60 221245.00
14                 29984   36.46   30.00 1093231.00    29984 117.08 115.50  3510566.20        29984  3.78   3.70 113289.80
15                  9882   56.27   45.00  556022.00     9882 433.01 426.20  4279016.70         9882 13.04  12.90 128830.50
16                 12123   50.60   40.00  613434.00    12123 287.19 286.50  3481650.00        12123 26.31  26.20 318965.10
17                  3478   61.19   40.00  212808.00     3478 908.71 861.00  3160509.10         3478  5.64   5.90  19619.10
18                 15535   46.96   35.00  729564.00    15535 208.64 208.40  3241241.50        15535 17.67  17.60 274457.40
19                  2031   68.14   45.00  138399.00     2031 761.37 698.80  1546348.00         2031 27.06  26.90  54949.30
20                  2968  446.54  435.00 1325320.00     2968 342.27 335.50  1015843.10         2968 11.32  10.90  33596.30
21                   443 1572.56 1470.00  696645.50      443 352.06 281.70   155962.40          443 29.67  29.10  13142.90
22                 22714   40.34   30.00  916211.00    22714 126.82 123.00  2880498.70        22714  6.87   6.80 156144.30
23                 27449   48.22   40.00 1323668.00    27449 308.37 305.00  8464418.50        27449  4.96   4.90 136031.30
24                  6547  268.39  255.00 1757166.00     6547 177.82 174.20  1164155.60         6547  2.85   2.70  18639.60
25                  3685   68.15   45.00  251127.00     3685 774.62 738.60  2854474.20         3685 13.72  13.40  50568.30
26                 22756   50.79   40.00 1155870.00    22756 442.27 438.60 10064395.00        22756  2.45   2.50  55711.30
27                  1399  479.32  480.00  670575.00     1399 323.62 305.40   452747.00         1399 24.23  23.90  33894.70
28                  3914   75.64   55.00  296057.00     3914 520.48 486.75  2037139.90         3914 36.20  36.20 141667.80
29                 15749   56.01   45.00  882102.00    15749 323.05 320.10  5087691.20        15749  8.70   8.60 137069.10
30                 32289   29.43   20.00  950410.00    32289  52.68  53.50  170

In [17]:
numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
assert 'kmeans_cluster' in numeric_columns, 'Expected kmeans_cluster to be a numeric grouping column.'
numeric_columns.remove('kmeans_cluster')
assert len(numeric_columns) >= 5, 'Expected at least five numeric feature columns for this aggregation test.'

agg_result = df.groupby('kmeans_cluster').agg({
    numeric_columns[1]: ['count', 'mean', 'median', 'sum'],
    numeric_columns[2]: ['count', 'mean', 'median', 'sum'],
    numeric_columns[3]: ['count', 'mean', 'median', 'sum'],
    numeric_columns[4]: ['count', 'mean', 'median', 'sum'],
})
assert not agg_result.empty, 'Grouped aggregation should produce a non-empty result.'
assert set(['count', 'mean', 'median', 'sum']).issubset(set(agg_result.columns.get_level_values(1)))

try:
    df.groupby('kmeans_cluster').agg({
        1: ['count', 'mean', 'median', 'sum'],
        2: ['count', 'mean', 'median', 'sum'],
        3: ['count', 'mean', 'median', 'sum'],
        4: ['count', 'mean', 'median', 'sum'],
    })
except KeyError:
    pass
else:
    raise AssertionError('Bug regression: integer-labeled aggregation keys unexpectedly succeeded.')